# ДЗ Семинар №8. Программный доступ к БД из Python
# Выполнил Чесноков Данила ДЭ 16-25

In [1]:
# Проверка окружения и общая конфигурация
import sys
print("Python:", sys.version.split()[0])

import psycopg, sqlalchemy, pandas, numpy
print("psycopg     :", psycopg.__version__)
print("sqlalchemy  :", sqlalchemy.__version__)
print("pandas      :", pandas.__version__)
print("numpy       :", numpy.__version__)

# Параметры подключения (внутри контейнера localhost = этот же контейнер).
# PostgreSQL настроен с trust-аутентификацией для локальных подключений,
# поэтому пароль НЕ требуется и НЕ передаётся.
DB = {
    "host":   "localhost",
    "port":   5432,
    "dbname": "demo",
    "user":   "postgres",
}

PSYCOPG_CONN = (
    f"host={DB['host']} port={DB['port']} dbname={DB['dbname']} "
    f"user={DB['user']}"
)
SQLALCHEMY_URL = (
    f"postgresql+psycopg://{DB['user']}@{DB['host']}:{DB['port']}/{DB['dbname']}"
)

print("\npsycopg DSN      :", PSYCOPG_CONN)
print("SQLAlchemy URL   :", SQLALCHEMY_URL)


Python: 3.11.17
psycopg     : 3.3.6
sqlalchemy  : 2.1.6
pandas      : 3.1.0
numpy       : 2.4.6

psycopg DSN      : host=localhost port=5432 dbname=demo user=postgres
SQLAlchemy URL   : postgresql+psycopg://postgres@localhost:5432/demo


## 1. Основы `psycopg` (DB-API)

**DB-API** (PEP 249) — стандарт Python для работы с реляционными БД: `connect()`, `cursor()`, `execute()`, `fetch*()`. У каждого драйвера свой диалект, но интерфейс общий.

Пакет `psycopg` (v3) — современный драйвер PostgreSQL. Что важно знать:

- **Соединение** открывается через `psycopg.connect(conn_string)`; его **обязательно** закрывать. Лучший способ — менеджер контекста `with psycopg.connect(...) as conn`.
- **Курсор** — объект, через который выполняются запросы; тоже умеет `with`.
- `cur.execute(sql, params)` — два аргумента: SQL-шаблон и кортеж/список параметров. Драйвер сам корректно экранирует и передаёт значения по бинарному протоколу. **Никогда не склеивайте SQL строками**.
- `cur.fetchone()` / `cur.fetchall()` / итерация по `cur` — способы получить результат.
- `psycopg.connect(..., row_factory=dict_row)` — строки как словари вместо кортежей.
- `psycopg.connect(...)` при выходе из `with` вызывает `close()`. Обратите внимание: `commit/rollback` на этом уровне **не** делается — это уже часть транзакций (см. тему 2).
- В реальных веб-приложениях напрямую `connect` почти не используют — берут пул (`psycopg_pool.ConnectionPool` или `QueuePool` в SQLAlchemy), потому что открытие нового TCP-соединения и форк бэкенд-процесса PG — дорого.


In [2]:
import psycopg
from psycopg.rows import dict_row

# --- 1. Версия сервера и типичный SELECT ---
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version()")
        print("PG version:", cur.fetchone()[0].split(" on ")[0])

# --- 2. DDL + DML + чтение в одной транзакции ---
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS demo_items")
        cur.execute("CREATE TABLE demo_items (id INT PRIMARY KEY, name TEXT, qty INT)")
        cur.execute(
            "INSERT INTO demo_items (id, name, qty) VALUES (%s, %s, %s), (%s, %s, %s)",
            (1, "apple", 10, 2, "banana", 20),
        )
    conn.commit()

    with conn.cursor() as cur:
        cur.execute("SELECT id, name, qty FROM demo_items ORDER BY id")
        print("\nrows (tuples):")
        for row in cur.fetchall():
            print(" ", row)

# --- 3. То же самое, но со словарями ---
with psycopg.connect(PSYCOPG_CONN, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT id, name, qty FROM demo_items ORDER BY id")
        print("\nrows (dict_row):")
        for row in cur.fetchall():
            print(" ", row)


PG version: PostgreSQL 15.19 (Debian 15.19-0+deb12u1)

rows (tuples):
  (1, 'apple', 10)
  (2, 'banana', 20)

rows (dict_row):
  {'id': 1, 'name': 'apple', 'qty': 10}
  {'id': 2, 'name': 'banana', 'qty': 20}


### Упражнение 1. Параметризованный поиск

**Задание.** В таблице `demo_items` найдите запись, где `name` совпадает с `target_name`, и выведите `id, name, qty`.

**Шаги.**
1. Откройте соединение `psycopg.connect(PSYCOPG_CONN)` и курсор.
2. Выполните `SELECT id, name, qty FROM demo_items WHERE name = %s` с параметром `(target_name,)`.
3. Напечатайте результат.
4. Дополнительно: поменяйте `target_name` на `"apple' OR '1'='1"` и убедитесь, что запрос вернул 0 строк — инъекция не сработала.

**Критерий успеха.** При `target_name = 'apple'` печатается `(1, 'apple', 10)`; при «инъекционном» вводе — пусто.


In [3]:
# УПРАЖНЕНИЕ 1. Параметризованный поиск по таблице demo_items
#
# Что делаем: ищем строки по точному совпадению name и проверяем, что «инъекционная»
# строка не меняет смысл запроса.
#
# Общая идея: SQL-текст и значения передаются ОТДЕЛЬНО. psycopg 3 по умолчанию использует
# серверную привязку параметров (extended protocol): текст запроса с плейсхолдерами уходит
# на сервер как есть, а значения — отдельным сообщением. Поэтому значение никогда не
# разбирается парсером SQL и не может стать «кодом».

def find_items(name):
    """Возвращает список кортежей (id, name, qty), где name равен переданному значению."""

    # psycopg.connect(PSYCOPG_CONN) — параметров, кроме строки подключения, не передаём:
    #  * row_factory не задан -> по умолчанию строки приходят КОРТЕЖАМИ (если передать
    #    row_factory=dict_row, придут словари {"id": 1, ...});
    #  * autocommit не задан -> по умолчанию False: psycopg сам открывает транзакцию
    #    (BEGIN) перед первым запросом и держит её открытой до commit()/rollback().
    #
    # Изоляция транзакции: isolation_level не задан, значит действует настройка сервера
    # default_transaction_isolation. В PostgreSQL это READ COMMITTED — каждый оператор
    # видит данные, зафиксированные на момент его начала. Для одного SELECT без
    # конкурентных записей этого достаточно, повышать уровень незачем.
    #
    # Блок `with psycopg.connect(...) as conn` в psycopg 3 на выходе:
    #  * без исключения -> COMMIT (у нас там только чтение, фиксировать нечего,
    #    но транзакция всё равно корректно завершается);
    #  * при исключении -> ROLLBACK;
    #  * и в обоих случаях ЗАКРЫВАЕТ соединение.
    with psycopg.connect(PSYCOPG_CONN) as conn:

        # Курсор — объект, через который выполняются запросы. Блок `with` закрывает
        # только курсор (освобождает ресурсы), коммит он НЕ делает.
        with conn.cursor() as cur:
            # Именованный стиль %(n)s: параметры передаются словарём. Эквивалент
            # позиционного %s + кортеж; именованный удобнее, когда параметр
            # встречается в запросе несколько раз или их много.
            # Плейсхолдер БЕЗ кавычек: кавычки вокруг значения ставить не нужно и нельзя —
            # значение вообще не подставляется в текст запроса.
            cur.execute(
                "SELECT id, name, qty FROM demo_items WHERE name = %(n)s",
                {"n": name},
            )
            # fetchall() забирает все строки в список. Для маленькой выборки это нормально,
            # для больших лучше итерироваться по курсору или брать fetchmany().
            return cur.fetchall()

# Два сценария: штатный ввод и попытка инъекции.
for candidate in ("apple", "apple' OR '1'='1"):
    found = find_items(candidate)
    print(f"ввод={candidate!r:22} -> найдено {len(found)}: {found}")

# Ожидаемый результат: для 'apple' — [(1, 'apple', 10)], для инъекции — [].
# Строка "apple' OR '1'='1" целиком сравнивается с колонкой name как обычный текст,
# а такого имени в таблице нет. Условие OR '1'='1' не сработало, потому что кавычки
# внутри значения не воспринимаются как часть SQL.



ввод='apple'                -> найдено 1: [(1, 'apple', 10)]
ввод="apple' OR '1'='1"     -> найдено 0: []


## 2. Транзакции и `with conn:`

- В PostgreSQL транзакция начинается с первого SQL-запроса.
- В psycopg3 у соединения есть метод `conn.transaction()`, но проще — **менеджер контекста `with conn:`**. Он делает `commit()` при успешном выходе и `rollback()` при любом исключении.
- **DDL транзакционен** (в отличие, например, от MySQL). Если вы упадёте после `CREATE TABLE` — таблица откатится.
- **Долгие транзакции — зло.** Пока транзакция открыта, PG не может почистить старые версии строк через `autovacuum` (MVCC, семинар 5). Отсюда bloat. Правило: транзакция должна длиться миллисекунды, а не минуты. Не держите транзакцию, пока ждёте ответа от медленного API.


In [4]:
from psycopg.errors import UniqueViolation

# Готовим таблицу ВНЕ проблемной транзакции
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS tx_demo")
        cur.execute("CREATE TABLE tx_demo (id INT PRIMARY KEY, name TEXT)")
    conn.commit()

try:
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            # Дубликат PK -> вторая строка упадёт
            cur.executemany("INSERT INTO tx_demo VALUES (%s, %s)", [(1, "a"), (1, "b")])
        conn.commit()
except UniqueViolation as e:
    print("Поймали UniqueViolation:", str(e).splitlines()[0])
    # with conn: автоматически сделает rollback

# Проверяем: в таблице 0 строк (вставка откатилась целиком)
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT count(*) FROM tx_demo")
        print("rows after rollback:", cur.fetchone()[0])


Поймали UniqueViolation: duplicate key value violates unique constraint "tx_demo_pkey"
rows after rollback: 0


### Упражнение 2. Атомарность при ошибке

**Задание.** Смоделируйте ситуацию, где часть строк вставляется успешно, а затем происходит ошибка, и убедитесь, что откатилось всё.

**Шаги.**
1. В отдельной транзакции создайте таблицу `ex2 (id INT PRIMARY KEY, name TEXT)` — без дубликатов.
2. В следующей транзакции выполните `executemany` со списком `[(1,'a'), (2,'b'), (1,'dup')]`.
3. Оберните в `try/except UniqueViolation`.
4. Проверьте `SELECT count(*) FROM ex2` — должно быть 0 (откатились даже успешные вставки).

**Критерий успеха.** Счётчик 0, в консоли — сообщение об ошибке.


In [5]:
# УПРАЖНЕНИЕ 2. Атомарность: ошибка в середине пакета откатывает ВСЕ вставки
#
# Сценарий: в пакете две корректные строки и дубликат первичного ключа. Ожидаем, что
# в таблице не останется ничего: транзакция либо проходит целиком, либо откатывается целиком.

def count_rows(table):
    """Отдельное соединение = отдельная короткая транзакция только на чтение."""
    with psycopg.connect(PSYCOPG_CONN) as c, c.cursor() as cur:
        # Имя таблицы подставляем f-строкой ТОЛЬКО потому что оно задано нами в коде.
        # Плейсхолдеры %s работают для значений, но не для имён таблиц/колонок.
        # Имя от пользователя так подставлять нельзя (нужен psycopg.sql.Identifier).
        cur.execute(f"SELECT count(*) FROM {table}")
        return cur.fetchone()[0]

# --- Транзакция №1: только DDL ---------------------------------------------------
# Создаём таблицу в отдельной транзакции, чтобы её существование не зависело от
# результата следующего шага. В PostgreSQL DDL транзакционен: если бы CREATE TABLE
# стоял в той же транзакции, что и упавшая вставка, таблица откатилась бы вместе с ней
# и проверять count(*) было бы не в чем.
# conn.execute(...) — сокращение: создаёт временный курсор и выполняет запрос.
# Транзакция закрывается на выходе из with (COMMIT, ошибок нет). Уровень изоляции
# по умолчанию — READ COMMITTED; для DDL без конкурентов он ничего не меняет.
with psycopg.connect(PSYCOPG_CONN) as conn:
    conn.execute("DROP TABLE IF EXISTS ex2")
    conn.execute("CREATE TABLE ex2 (id INT PRIMARY KEY, name TEXT)")
print("строк после создания таблицы:", count_rows("ex2"))   # 0

# --- Транзакция №2: пакет вставок с дубликатом -----------------------------------
batch = [(1, "a"), (2, "b"), (1, "dup")]   # третья строка нарушает PRIMARY KEY (id=1 уже есть)
try:
    # autocommit по умолчанию False, поэтому все три INSERT выполняются в ОДНОЙ
    # транзакции, которую psycopg открыл неявно. Именно это даёт атомарность:
    # фиксация происходит только если весь блок завершился без исключения.
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            # executemany отправляет один и тот же INSERT для каждого кортежа
            # (в psycopg >= 3.1 — через pipeline-режим, без ожидания ответа на каждую строку).
            # На третьей строке сервер вернёт ошибку 23505, и транзакция перейдёт
            # в состояние «aborted»: любые следующие команды в ней вернут ошибку,
            # пока не будет ROLLBACK.
            cur.executemany("INSERT INTO ex2 (id, name) VALUES (%s, %s)", batch)
        # Сюда выполнение не дойдёт, COMMIT не будет.
except UniqueViolation as err:
    # Исключение вылетело из блока with conn -> psycopg выполнил ROLLBACK и закрыл
    # соединение. Транзакция закрыта именно здесь, до начала except-ветки.
    # sqlstate — стандартный пятизначный код ошибки; 23505 = unique_violation.
    print("Поймали:", type(err).__name__, "| SQLSTATE =", err.sqlstate)

# --- Проверка ---------------------------------------------------------------------
# Читаем новой транзакцией. Строки (1,'a') и (2,'b') вставились успешно, но не были
# зафиксированы, поэтому откатились вместе с остальными: ожидаем 0.
print("строк после неудачного пакета:", count_rows("ex2"))


строк после создания таблицы: 0
Поймали: UniqueViolation | SQLSTATE = 23505
строк после неудачного пакета: 0


## 3. SQL-инъекции и параметризация

**SQL-инъекция** — это когда пользовательский ввод попадает в строку SQL и интерпретируется как код.

- ❌ **Уязвимо:** `cur.execute(f"... WHERE name = '{name}'")`. Если `name = "' OR '1'='1"`, запрос вернёт все строки; если `name = "'; DROP TABLE ...; --"` — таблица исчезнет.
- ✅ **Безопасно:** `cur.execute("... WHERE name = %s", (name,))`. Данные и SQL идут раздельно, значения передаются по бинарному протоколу и **не могут быть интерпретированы как код**.
- То же в SQLAlchemy Core: `text("... WHERE name = :name")` + `{"name": ...}`. **Никогда** не подставляйте значения внутрь `text()` через f-строку.


In [6]:
def unsafe_search(name):
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            # ❌ f-string — SQL-инъекция
            cur.execute(f"SELECT id, name, qty FROM demo_items WHERE name = '{name}'")
            return cur.fetchall()

def safe_search(name):
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            # ✅ параметризация
            cur.execute("SELECT id, name, qty FROM demo_items WHERE name = %s", (name,))
            return cur.fetchall()

malicious = "' OR '1'='1' --"
print("unsafe :", unsafe_search(malicious))
print("safe   :", safe_search(malicious))


unsafe : [(1, 'apple', 10), (2, 'banana', 20)]
safe   : []


### Упражнение 3. Почините уязвимый запрос

**Задание.** Дана функция `broken(code)` — она ищет пилота по коду через f-string. Перепишите её безопасно.

**Шаги.**
1. Скопируйте функцию, замените f-string на `%s` + параметр.
2. Проверьте: при `code = "' OR '1'='1"` результат должен быть пустым.
3. Дополнительно: добавьте `LIMIT 5` в SQL.

**Критерий успеха.** `broken("' OR '1'='1")` (старая версия) вернёт всё, `fixed("' OR '1'='1")` — пусто.


In [7]:
# УПРАЖНЕНИЕ 3. Исправляем SQL-инъекцию в функции broken()
#
# Как возникает уязвимость: f-строка подставляет ввод пользователя в текст запроса ДО
# отправки на сервер, и парсер PostgreSQL видит его как часть SQL-кода.

def broken(code):
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            # УЯЗВИМО: при code = "' OR '1'='1" получится
            #   ... WHERE name = '' OR '1'='1'
            # Условие '1'='1' всегда истинно, поэтому вернутся все строки.
            cur.execute(f"SELECT id, name FROM demo_items WHERE name = '{code}'")
            return cur.fetchall()

def fixed(code):
    # Параметры по умолчанию те же, что в упражнении 1: autocommit=False, значит запрос
    # идёт в неявной транзакции (READ COMMITTED), которая закрывается на выходе из
    # `with conn` (COMMIT, ошибок нет) вместе с соединением.
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            cur.execute(
                # %s — плейсхолдер: значение code передаётся отдельно от текста запроса.
                # ORDER BY нужен, чтобы LIMIT возвращал предсказуемый набор: без
                # сортировки порядок строк в PostgreSQL не гарантирован.
                # LIMIT 5 — защитное ограничение, даже при ошибке в фильтре
                # приложение не получит всю таблицу.
                "SELECT id, name FROM demo_items WHERE name = %s ORDER BY id LIMIT 5",
                (code,),   # кортеж из одного элемента: запятая обязательна, иначе это просто строка в скобках
            )
            return cur.fetchall()

attack = "' OR '1'='1"
print("broken(атака):", broken(attack))    # утечка: вернулись все строки таблицы
print("fixed(атака)  :", fixed(attack))    # []: строка сравнивается как обычный текст
print("fixed('apple'):", fixed("apple"))   # штатный ввод работает как прежде


broken(атака): [(1, 'apple'), (2, 'banana')]
fixed(атака)  : []
fixed('apple'): [(1, 'apple')]


## 4. Батчевая вставка: `executemany`

- Цикл `for row in rows: cur.execute("INSERT ...", row)` — это **N сетевых round-trip'ов** и N записей в WAL. Медленно.
- `cur.executemany(sql, list_of_tuples)` — один вызов, драйвер группирует строки в пакеты.
- В `psycopg3` (≥3.1) `executemany` дополнительно объединяет несколько `INSERT` в один `pipeline`-пакет.
- Для по-настоящему больших объёмов (миллионы строк) есть отдельный, ещё более быстрый протокол — `COPY` (`cur.copy()`), но `executemany` — уже огромный шаг вперёд по сравнению с циклом.


In [8]:
import time, random

N = 3000
rows = [(i, f"item_{i}", random.randint(1, 100)) for i in range(1, N + 1)]

# Готовим таблицу
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS bench")
        cur.execute("CREATE TABLE bench (id INT PRIMARY KEY, name TEXT, val INT)")
    conn.commit()

# 1) Цикл
t0 = time.perf_counter()
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        for r in rows:
            cur.execute("INSERT INTO bench VALUES (%s, %s, %s)", r)
    conn.commit()
t_loop = time.perf_counter() - t0
print(f"цикл         : {t_loop:.2f}s")

# 2) executemany
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("TRUNCATE bench")
    conn.commit()

t0 = time.perf_counter()
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.executemany("INSERT INTO bench VALUES (%s, %s, %s)", rows)
    conn.commit()
t_batch = time.perf_counter() - t0
print(f"executemany  : {t_batch:.2f}s")
print(f"ускорение    : x{t_loop / max(t_batch, 1e-9):.1f}")


цикл         : 0.11s
executemany  : 0.02s
ускорение    : x6.7


### Упражнение 4. Батч на 10 000 строк

**Задание.** Сравните цикл и `executemany` на 10 000 строк и найдите порог, на котором `executemany` стабильно быстрее.

**Шаги.**
1. Сгенерируйте 10 000 кортежей `(id, name, val)`.
2. Замерьте цикл, замерьте `executemany`.
3. Напечатайте `speedup = t_loop / t_batch`.
4. **Вопрос на подумать:** почему `executemany` не «в N раз быстрее» линейно?

**Критерий успеха.** `speedup > 5`. В комментарии — 1–2 строки объяснения.


In [9]:
# УПРАЖНЕНИЕ 4. Цикл execute против executemany на 10 000 строк
import time, random

N4 = 10_000
# random.Random(8) — отдельный генератор с фиксированным seed. Данные одинаковы
# при каждом запуске, а глобальное состояние random не меняется.
rng = random.Random(8)
rows4 = [(i, f"unit_{i:05d}", rng.randint(1, 500)) for i in range(N4)]

def reset_table():
    # Каждый замер начинаем с пустой таблицы: без этого второй замер упал бы на
    # дубликатах PK. Отдельная транзакция; закрывается на выходе из with (COMMIT).
    with psycopg.connect(PSYCOPG_CONN) as conn:
        conn.execute("DROP TABLE IF EXISTS bench_ex4")
        conn.execute("CREATE TABLE bench_ex4 (id INT PRIMARY KEY, name TEXT, val INT)")

def run_loop(rs):
    # Вариант 1: N отдельных execute. autocommit=False (по умолчанию), поэтому все
    # вставки идут в ОДНОЙ транзакции с одним COMMIT на выходе из with. Так сравнение
    # честное: разница между вариантами — это сетевые обращения, а не число fsync.
    # (С autocommit=True каждая вставка фиксировалась бы отдельно, и цикл был бы
    # ещё в разы медленнее из-за записи WAL на диск при каждом коммите.)
    # Изоляция — READ COMMITTED, по умолчанию.
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            for r in rs:
                cur.execute("INSERT INTO bench_ex4 VALUES (%s, %s, %s)", r)

def run_many(rs):
    # Вариант 2: один вызов executemany. Транзакция та же: одна, COMMIT на выходе из with.
    # Выигрыш даёт pipeline-режим psycopg >= 3.1: запросы отправляются пачкой, и драйвер
    # не ждёт подтверждения каждой строки, как в цикле (там ~N сетевых round-trip).
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            cur.executemany("INSERT INTO bench_ex4 VALUES (%s, %s, %s)", rs)

def timed(fn, rs):
    """Пересоздаёт таблицу и возвращает время работы fn(rs) в секундах.
    perf_counter — монотонные часы высокого разрешения, подходят для замеров интервалов."""
    reset_table()
    t0 = time.perf_counter()
    fn(rs)
    return time.perf_counter() - t0

t_loop4 = timed(run_loop, rows4)
t_many4 = timed(run_many, rows4)
print(f"10 000 строк: цикл {t_loop4:.2f} c | executemany {t_many4:.2f} c | speedup = x{t_loop4 / t_many4:.1f}")

# Порог выгоды: повторяем замер на разных размерах батча.
# Время подключения и создания таблицы входит в замер только для reset_table(),
# который стоит ДО t0, поэтому на результат он не влияет.
for n in (10, 100, 1000, 5000):
    sub = rows4[:n]
    a, b = timed(run_loop, sub), timed(run_many, sub)
    print(f"n={n:>5}: цикл {a:.3f} c | executemany {b:.3f} c | x{a / max(b, 1e-9):.1f}")

# ПОЧЕМУ УСКОРЕНИЕ НЕ РАВНО N:
# 1) executemany убирает только сетевые round-trip'ы. Сервер всё равно разбирает и
#    исполняет каждую вставку, обновляет индекс первичного ключа и пишет WAL.
# 2) Фиксированные расходы (подключение, COMMIT с fsync) одинаковы в обоих вариантах
#    и на малых батчах составляют основную часть времени, поэтому на n=10 разница мала.
# 3) На localhost задержка сети почти нулевая; на удалённом сервере выигрыш был бы больше.
# Для миллионов строк быстрее всего COPY (cur.copy()): один поток данных вместо INSERT'ов.


10 000 строк: цикл 0.37 c | executemany 0.05 c | speedup = x7.1
n=   10: цикл 0.003 c | executemany 0.003 c | x1.1
n=  100: цикл 0.005 c | executemany 0.003 c | x1.7
n= 1000: цикл 0.031 c | executemany 0.007 c | x4.4
n= 5000: цикл 0.197 c | executemany 0.025 c | x7.8


## 5. SQLAlchemy Core

**Core** — это слой выражений SQL поверх DB-API. Даёт:

- **Пул соединений** из коробки (`QueuePool`).
- **Параметризацию:** `text("WHERE x = :x")` + `[{...}, {...}]` — автоматически превращается в `executemany`.
- Единый интерфейс к PostgreSQL / SQLite / MySQL.

Ключевые вещи:
- `create_engine(url, echo=False)` — движок, управляет пулом. `echo=True` печатает SQL в консоль (обязательно при отладке ORM).
- `text("SELECT ... :param")` — обёртка над строкой SQL. Параметры — именованные `:name`. **f-строки внутрь `text()` нельзя.**
- `engine.connect()` → `conn.execute(stmt, params)` → `conn.commit()`.
- Если второй аргумент `execute` — список словарей, Core сам выполнит `executemany`.


In [10]:
from sqlalchemy import create_engine, text

engine = create_engine(SQLALCHEMY_URL, echo=False)

# --- DDL + батчевая вставка через Core ---
with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS core_demo"))
    conn.execute(text("CREATE TABLE core_demo (id INT PRIMARY KEY, sensor TEXT, val NUMERIC)"))
    conn.commit()

    records = [
        {"id": i, "s": f"SENSOR_{i % 7}", "v": round((i * 0.37) % 100, 2)}
        for i in range(1, 501)
    ]
    # список dict -> автоматически executemany
    conn.execute(
        text("INSERT INTO core_demo (id, sensor, val) VALUES (:id, :s, :v)"),
        records,
    )
    conn.commit()
    print(f"Вставлено {len(records)} строк через Core.executemany")

    # Простой агрегат через Core
    row = conn.execute(
        text("SELECT count(*), max(val) FROM core_demo WHERE val > :threshold"),
        {"threshold": 50},
    ).one()
    print(f"Строк с val > 50: {row[0]}, максимум: {row[1]}")


Вставлено 500 строк через Core.executemany
Строк с val > 50: 230, максимум: 99.9


### Упражнение 5. Core: создать таблицу и залить батч

**Задание.** Через `text()` и `engine.connect()` создайте таблицу `ex5_measurements (id, sensor, value)` и вставьте 200 строк списком словарей.

**Шаги.**
1. `DROP TABLE IF EXISTS ex5_measurements` и `CREATE TABLE ...`.
2. Сгенерируйте `records` — список словарей `{"id", "sensor", "value"}` на 200 элементов.
3. Один вызов `conn.execute(text("INSERT ..."), records)`.
4. Проверьте: `SELECT count(*) FROM ex5_measurements WHERE value > 50`.

**Критерий успеха.** В консоли: `count > 0`. В логах — один `INSERT` (executemany), не 200.


In [11]:
# УПРАЖНЕНИЕ 5. SQLAlchemy Core: создать таблицу и залить 200 строк одним вызовом
#
# `engine` создан выше как create_engine(SQLALCHEMY_URL, echo=False):
#  * echo=False (по умолчанию) — SQL в консоль не печатается. С echo=True SQLAlchemy
#    логировал бы каждый запрос — удобно, чтобы убедиться, что INSERT ушёл пачкой.
#  * пул соединений — QueuePool (по умолчанию: 5 постоянных соединений + до 10 временных);
#    при выходе из блока соединение возвращается в пул, а не закрывается физически.

SENSORS = ["alpha", "beta", "gamma", "delta", "epsilon"]
records5 = [
    # Список словарей: ключи совпадают с именованными параметрами :id, :sensor, :value.
    {"id": n, "sensor": SENSORS[n % len(SENSORS)], "value": round(n * 0.53, 2)}
    for n in range(1, 201)
]

# engine.begin() — контекстный менеджер «одна транзакция»:
#  * транзакция начинается при входе в блок;
#  * при нормальном выходе — COMMIT, при исключении — ROLLBACK;
#  * затем соединение возвращается в пул.
# Это надёжнее, чем engine.connect() + ручной conn.commit(): забыть commit невозможно.
# (У engine.connect() в SQLAlchemy 2.x автокоммита нет: без явного commit() изменения
# при выходе откатываются.) Изоляция — по умолчанию та, что у сервера: READ COMMITTED.
# Параметр create_engine(isolation_level="...") позволил бы задать другой.
# DDL и вставку держим в ОДНОЙ транзакции: либо таблица создана и заполнена целиком,
# либо не осталось ничего.
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS ex5_measurements"))
    conn.execute(text(
        "CREATE TABLE ex5_measurements (id INT PRIMARY KEY, sensor TEXT, value NUMERIC)"
    ))
    # text() — «сырой» SQL; параметры задаются как :имя и передаются отдельно от текста,
    # так что инъекция невозможна. Если вторым аргументом передать СПИСОК словарей,
    # Core выполнит executemany: один вызов вместо 200 отдельных execute.
    conn.execute(
        text("INSERT INTO ex5_measurements (id, sensor, value) VALUES (:id, :sensor, :value)"),
        records5,
    )
    # Проверяем вставку внутри этой же транзакции: она видит свои нефиксированные
    # изменения. rowcount для executemany зависит от драйвера, поэтому считаем count(*).
    inserted = conn.execute(text("SELECT count(*) FROM ex5_measurements")).scalar_one()
    print("вставлено строк:", inserted)   # 200
# Здесь транзакция зафиксирована (COMMIT), соединение вернулось в пул.

# Новая короткая транзакция только на чтение. engine.connect() сам ничего не фиксирует:
# для SELECT это не нужно, при выходе из блока транзакция откатывается (изменений нет).
with engine.connect() as conn:
    big = conn.execute(
        text("SELECT count(*) FROM ex5_measurements WHERE value > :limit"),
        {"limit": 50},                    # для одного набора параметров достаточно словаря
    ).scalar_one()                         # scalar_one(): ровно одна строка и одна колонка, иначе ошибка
    print("значений > 50:", big)           # ожидаем > 0


вставлено строк: 200
значений > 50: 106


## 6. Pandas + `chunksize`

`pandas` — in-memory аналитика. **DataFrame** — это таблица в RAM.

- `pd.read_sql(query, engine, chunksize=500)` возвращает **итератор** по чанкам. Так вы не выгрузите 10 ГБ в 8 ГБ RAM.
- `np.where(cond, a, b)` — векторизованный `CASE WHEN`. Работает на C, а не в Python-цикле.
- `chunk[chunk['col'] > 0]` — фильтрация тоже векторизована.

**Разделение труда:** PostgreSQL — для OLTP и сложных `JOIN`. Pandas — для математических трансформаций, feature engineering, агрегатов «в памяти».


In [12]:
import pandas as pd
import numpy as np

total = {"HIGH": 0, "LOW": 0}
for chunk in pd.read_sql("SELECT * FROM core_demo", engine, chunksize=200):
    chunk["level"] = np.where(chunk["val"] > 50, "HIGH", "LOW")
    vc = chunk["level"].value_counts()
    for k in total:
        total[k] += int(vc.get(k, 0))
print("По всей таблице:", total)
print("Строк всего   :", sum(total.values()))


По всей таблице: {'HIGH': 230, 'LOW': 270}
Строк всего   : 500


### Упражнение 6. Агрегат по сенсорам

**Задание.** Читая `ex5_measurements` чанками по 50, посчитайте среднее `value` и количество замеров по каждому `sensor`.

**Шаги.**
1. Итерируйтесь `pd.read_sql(..., engine, chunksize=50)`.
2. Для каждого чанка считайте `groupby("sensor").agg(mean=("value","mean"), n=("value","size"))`.
3. Соберите результаты в словарь `{sensor: {"sum": ..., "n": ...}}` (агрегируйте среднее, пересчитывая взвешенно).
4. Напечатайте итоговую таблицу.

**Критерий успеха.** Сумма `n` по всем сенсорам == общее число строк в `ex5_measurements`.


In [13]:
# УПРАЖНЕНИЕ 6. Среднее и количество замеров по сенсорам, чтение чанками
from collections import defaultdict

# Идея: таблицу читаем порциями по 50 строк, а итог собираем из частичных агрегатов.
# Среднее средних нельзя: чанки могут быть разного размера и сенсоры в них представлены
# неравномерно. Поэтому накапливаем сумму и количество, а среднее считаем один раз в конце.
acc = defaultdict(lambda: {"sum": 0.0, "n": 0})

# pd.read_sql(query, engine, chunksize=50):
#  * без chunksize (по умолчанию None) вернулся бы один DataFrame со ВСЕМИ строками;
#  * с chunksize вернётся итератор DataFrame по 50 строк, и в памяти pandas одновременно
#    лежит только один чанк.
# Оговорка: пока драйвер использует обычный клиентский курсор, он может загрузить весь
# результат целиком, а pandas лишь порежет его на чанки. Чтобы тянуть данные по частям
# с сервера, нужен потоковый курсор (execution_options(stream_results=True)).
# Транзакцию pandas открывает и закрывает сам на соединении из пула engine; наш код
# внутри цикла к БД не обращается.
query = "SELECT sensor, value FROM ex5_measurements"
for part in pd.read_sql(query, engine, chunksize=50):
    # NUMERIC из PostgreSQL приходит в Python как Decimal (dtype object).
    # astype(float) нужен, чтобы pandas мог суммировать и усреднять векторно.
    part["value"] = part["value"].astype(float)
    # groupby + agg по одной колонке: сумма и число значений для каждого сенсора в чанке.
    stats = part.groupby("sensor")["value"].agg(["sum", "count"])
    for sensor, r in stats.iterrows():
        acc[sensor]["sum"] += r["sum"]
        acc[sensor]["n"]   += int(r["count"])

summary = pd.DataFrame(
    [(s, a["sum"] / a["n"], a["n"]) for s, a in acc.items()],
    columns=["sensor", "mean", "n"],
).sort_values("sensor").reset_index(drop=True)
print(summary.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# Контроль целостности: сумма n по сенсорам должна совпасть с числом строк в таблице.
# Отдельная короткая транзакция только на чтение, изоляция READ COMMITTED по умолчанию.
with engine.connect() as conn:
    total_in_db = conn.execute(text("SELECT count(*) FROM ex5_measurements")).scalar_one()
print("сумма n:", int(summary["n"].sum()), "| строк в таблице:", total_in_db,
      "| совпало:", int(summary["n"].sum()) == total_in_db)


 sensor  mean  n
  alpha 54.33 40
   beta 52.20 40
  delta 53.27 40
epsilon 53.80 40
  gamma 52.73 40
сумма n: 200 | строк в таблице: 200 | совпало: True


## 7. SQLAlchemy ORM

ORM мапит таблицы на Python-классы. Удобно для CRUD, но требует дисциплины.

- Наследуемся от `DeclarativeBase`; колонки описываем через `Mapped[T] = mapped_column(...)`.
- **M:N** — через ассоциативную `Table`, в `relationship(secondary=assoc)`.
- **Каскад** — `ondelete='CASCADE'` **на уровне БД** (в `ForeignKey`). Это быстрее, чем ORM-каскад: PG удалит сам, без выборки всех дочерних строк в память.
- `with Session(engine) as s:` — сессия. `s.add(obj)`, `s.commit()`, `s.delete(obj)`, `s.scalars(select(...))`.
- `engine.echo = True` — печатать SQL в консоль (включайте при отладке!).


In [14]:
from sqlalchemy import ForeignKey, String, Table, Column, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session

class Base(DeclarativeBase):
    pass

pilot_aircraft = Table(
    "pilot_aircraft", Base.metadata,
    Column("pilot_id", ForeignKey("pilots.id", ondelete="CASCADE"), primary_key=True),
    Column("aircraft_code", ForeignKey("aircrafts_orm.aircraft_code", ondelete="CASCADE"), primary_key=True),
)

class Aircraft(Base):
    __tablename__ = "aircrafts_orm"
    aircraft_code: Mapped[str] = mapped_column(String(3), primary_key=True)
    model:         Mapped[str] = mapped_column(String(50))
    pilots: Mapped[list["Pilot"]] = relationship(secondary=pilot_aircraft, back_populates="aircrafts")

class Pilot(Base):
    __tablename__ = "pilots"
    id:   Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    aircrafts: Mapped[list["Aircraft"]] = relationship(secondary=pilot_aircraft, back_populates="pilots")

Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

with Session(engine) as s:
    boeing = Aircraft(aircraft_code="773", model="Boeing 777-300")
    p1 = Pilot(name="Иван", aircrafts=[boeing])
    s.add(p1)
    s.commit()

    # Каскадное удаление
    to_del = s.query(Pilot).filter_by(name="Иван").first()
    s.delete(to_del)
    s.commit()

    # Связей не осталось?
    links = s.execute(select(pilot_aircraft)).fetchall()
    print("Связей после каскада:", len(links))   # 0


Связей после каскада: 0


### Упражнение 7. Своя M:N-модель

**Задание.** Опишите M:N между `Author` и `Book` через ассоциативную `author_book`.

**Шаги.**
1. Объявите `Author` (id, name) и `Book` (id, title) и `author_book` через `Table`.
2. В `relationship` пропишите `secondary=author_book` и `back_populates` с обеих сторон.
3. `create_all`.
4. Добавьте 2 авторов и 3 книги (некоторые — с двумя авторами).
5. Удалите одного автора и проверьте, что связи в `author_book` для него исчезли, а книги, у которых были другие авторы, остались.

**Критерий успеха.** До удаления связей 3–4, после удаления — меньше на число связей удалённого автора.


In [15]:
# УПРАЖНЕНИЕ 7. Связь многие-ко-многим Author <-> Book через author_book
from sqlalchemy import func

# Отдельный Base (LibBase), чтобы метаданные не смешивались с моделями Pilot/Aircraft
# из демо выше: drop_all/create_all затронут только таблицы этой модели.
class LibBase(DeclarativeBase):
    pass

# Ассоциативная таблица описана через Table, без отдельного класса: своих полей у связи нет.
# Составной первичный ключ (author_id, book_id) запрещает дубликат пары «автор — книга».
# ondelete="CASCADE" — это правило В БАЗЕ ДАННЫХ (FOREIGN KEY ... ON DELETE CASCADE):
# при удалении автора или книги PostgreSQL сам удалит строки связи. Родительская сторона
# при этом не страдает: удаляются только строки author_book, но не книги.
author_book = Table(
    "author_book", LibBase.metadata,
    Column("author_id", ForeignKey("authors.id", ondelete="CASCADE"), primary_key=True),
    Column("book_id",   ForeignKey("books.id",   ondelete="CASCADE"), primary_key=True),
)

class Author(LibBase):
    __tablename__ = "authors"
    id:   Mapped[int] = mapped_column(primary_key=True)    # int PK => SERIAL/IDENTITY, значение даёт БД
    name: Mapped[str] = mapped_column(String(100))
    # secondary=author_book — ORM понимает, что связь идёт через ассоциативную таблицу.
    # back_populates — двусторонняя связь: добавил книгу автору, и автор появился в book.authors.
    # passive_deletes=True: при удалении автора ORM не загружает его книги, чтобы удалить
    # связи самому, а полагается на ON DELETE CASCADE в БД. По умолчанию (False) ORM
    # сначала подгрузит коллекцию и удалит строки связи сам. Итог одинаков, но
    # при passive_deletes=True лишнего SELECT нет.
    books: Mapped[list["Book"]] = relationship(
        secondary=author_book, back_populates="authors", passive_deletes=True
    )

class Book(LibBase):
    __tablename__ = "books"
    id:    Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(200))
    authors: Mapped[list["Author"]] = relationship(
        secondary=author_book, back_populates="books", passive_deletes=True
    )

# drop_all/create_all выполняются в собственных транзакциях: DDL в PostgreSQL
# транзакционен. По умолчанию checkfirst=True: не падаем, если таблицы нет или есть.
LibBase.metadata.drop_all(engine)
LibBase.metadata.create_all(engine)

def count_links(session):
    """Число строк в таблице связей."""
    return session.scalar(select(func.count()).select_from(author_book))

# Session(engine) — единица работы ORM. Транзакция открывается автоматически при первом
# обращении к БД (autobegin) и держится до commit()/rollback(). Уровень изоляции —
# серверный по умолчанию, READ COMMITTED. Блок `with` на выходе ЗАКРЫВАЕТ сессию, но НЕ
# коммитит: всё, что не зафиксировано через s.commit(), откатится.
with Session(engine) as s:
    ilf, petrov = Author(name="Ильф"), Author(name="Петров")
    # Объекты связаны через Python-списки; строки author_book ORM создаст сам.
    chairs = Book(title="Двенадцать стульев", authors=[ilf, petrov])   # два автора
    calf   = Book(title="Золотой телёнок",    authors=[ilf, petrov])   # два автора
    notes  = Book(title="Записные книжки",    authors=[ilf])           # один автор
    s.add_all([chairs, calf, notes])
    # Транзакция №1: INSERT авторов, книг и связей. COMMIT здесь.
    s.commit()
    print("связей до удаления:", count_links(s))   # 5: Ильф — 3, Петров — 2

    # После commit() объекты «просрочены» (expire_on_commit=True по умолчанию), и
    # обращение к их атрибутам запускает новую транзакцию с перечитыванием данных.
    victim = s.scalar(select(Author).where(Author.name == "Ильф"))
    s.delete(victim)
    # Транзакция №2: DELETE автора; строки связи удаляет БД (CASCADE). COMMIT здесь.
    s.commit()

    print("связей после удаления:", count_links(s))   # 2: остались две связи Петрова
    print("книг осталось:", s.scalar(select(func.count()).select_from(Book)))   # 3
    # «Записные книжки» остались без авторов: связь удалена, а сама книга нет,
    # потому что каскад идёт от автора только к строкам связи.
    for b in s.scalars(select(Book).order_by(Book.id)):
        print("  ", b.title, "->", [a.name for a in b.authors])
# Последняя (неявная) транзакция была только на чтение; при выходе из with сессия
# закрывается и транзакция откатывается — менять в ней было нечего.


связей до удаления: 5
связей после удаления: 2
книг осталось: 3
   Двенадцать стульев -> ['Петров']
   Золотой телёнок -> ['Петров']
   Записные книжки -> []


## 8. Проблема N+1 и `selectinload`

**N+1** — это когда вы делаете **1** запрос за родителями, а потом в цикле дёргаете `p.children` — и ORM делает ещё **N** запросов (по одному на родителя). Итого `N+1` сетевых обращений вместо двух.

Как поймать: `engine.echo=True` — в логах видно один `SELECT ... FROM parents` и N однотипных `SELECT ... FROM children WHERE parent_id = ...`.

Как лечить:
- `selectinload(Parent.children)` — ровно **2** запроса: `SELECT parents` + `SELECT children WHERE parent_id IN (...)`. Часто рекомендуемый дефолт.
- `joinedload(Parent.children)` — **1** запрос с `LEFT OUTER JOIN`, но с дублями родительских колонок. На широких таблицах — лишний трафик. Требует `.unique()` при использовании с `scalars().all()`.


In [16]:
from sqlalchemy import event
from sqlalchemy.orm import selectinload

def count_queries(fn):
    """Считает SQL-запросы, которые выполнит fn()."""
    counter = {"n": 0}
    def before(conn, cursor, statement, params, ctx, many):
        counter["n"] += 1
    event.listen(engine, "before_cursor_execute", before)
    try:
        result = fn()
    finally:
        event.remove(engine, "before_cursor_execute", before)
    return result, counter["n"]

# Готовим данные: 5 пилотов, у каждого 2 самолёта
with Session(engine) as s:
    s.query(Pilot).delete()
    s.query(Aircraft).delete()
    s.commit()
    for i in range(5):
        a1 = Aircraft(aircraft_code=f"A{i}1", model=f"Model {i}-1")
        a2 = Aircraft(aircraft_code=f"A{i}2", model=f"Model {i}-2")
        s.add(Pilot(name=f"Pilot_{i}", aircrafts=[a1, a2]))
    s.commit()

def lazy_load():
    with Session(engine) as s:
        pilots = s.scalars(select(Pilot)).all()
        for p in pilots:
            _ = p.aircrafts
        return len(pilots)

def eager_load():
    with Session(engine) as s:
        stmt = select(Pilot).options(selectinload(Pilot.aircrafts))
        pilots = s.scalars(stmt).all()
        for p in pilots:
            _ = p.aircrafts
        return len(pilots)

_, n_lazy  = count_queries(lazy_load)
_, n_eager = count_queries(eager_load)
print(f"ленивая (N+1)     : {n_lazy} запросов")
print(f"selectinload      : {n_eager} запросов")


ленивая (N+1)     : 6 запросов
selectinload      : 2 запросов


### Упражнение 8. `joinedload` и `.unique()`

**Задание.** Решите ту же задачу через `joinedload` и убедитесь, что запрос действительно один.

**Шаги.**
1. Импортируйте `joinedload` из `sqlalchemy.orm`.
2. Замените `selectinload(Pilot.aircrafts)` на `joinedload(Pilot.aircrafts)`.
3. Запустите `eager_load()` — получите `InvalidRequestError` (нужна дедупликация).
4. Исправьте: `s.scalars(stmt).unique().all()`.
5. Пересчитайте запросы через `count_queries`.

**Критерий успеха.** 1 запрос вместо 2. В комментарии — почему `selectinload` всё равно рекомендуют чаще.


In [17]:
# УПРАЖНЕНИЕ 8. joinedload и .unique(): тот же запрос за один SELECT
from sqlalchemy.orm import joinedload
from sqlalchemy.exc import InvalidRequestError

# Что делают стратегии загрузки коллекции Pilot.aircrafts:
#  * по умолчанию (lazy="select"): коллекция грузится при первом обращении p.aircrafts —
#    отдельным запросом на каждого пилота. Итого 1 + N (проблема N+1);
#  * selectinload: после основного запроса один дополнительный
#    SELECT ... WHERE pilot_id IN (...). Итого 2;
#  * joinedload: один запрос, в который добавлен LEFT OUTER JOIN (тип JOIN по
#    умолчанию — LEFT OUTER, пилот без самолётов не пропадёт). Итого 1.

def load_joined_no_unique():
    # Транзакция сессии: неявная, открывается первым запросом, READ COMMITTED
    # по умолчанию; закрывается на выходе из with (сессия закрыта, откат,
    # менять нечего).
    with Session(engine) as s:
        stmt = select(Pilot).options(joinedload(Pilot.aircrafts))
        # JOIN размножает пилота: на каждый его самолёт приходит отдельная строка результата.
        # Для коллекций SQLAlchemy 2.x требует явно вызвать .unique(), иначе
        # InvalidRequestError. Так ORM защищает от случайных дубликатов.
        return len(s.scalars(stmt).all())

def load_joined():
    with Session(engine) as s:
        stmt = select(Pilot).options(joinedload(Pilot.aircrafts))
        # .unique() схлопывает повторяющиеся объекты Pilot, пришедшие из разных
        # строк JOIN, так что каждый пилот встречается один раз.
        pilots = s.scalars(stmt).unique().all()
        # Коллекции уже загружены тем же запросом, поэтому доступ к ним не создаёт
        # новых SQL-запросов; сумма нужна лишь для проверки, что данные на месте.
        return sum(len(p.aircrafts) for p in pilots)

# 1) Демонстрируем ожидаемую ошибку без unique()
try:
    load_joined_no_unique()
except InvalidRequestError as err:
    print("без unique():", str(err).split(".")[0])

# 2) Считаем запросы для всех трёх стратегий. count_queries (из ячейки выше) вешает
#    слушатель before_cursor_execute на engine и считает SQL-операторы на время вызова.
total_aircrafts, q_joined = count_queries(load_joined)
_, q_lazy   = count_queries(lazy_load)      # ожидаем 1 + 5 = 6
_, q_select = count_queries(eager_load)     # ожидаем 2
print(f"lazy (N+1): {q_lazy} | selectinload: {q_select} | joinedload: {q_joined}")
print("самолётов загружено joinedload:", total_aircrafts)   # 10 (5 пилотов по 2 самолёта)

# ПОЧЕМУ selectinload РЕКОМЕНДУЮТ ЧАЩЕ:
# * JOIN повторяет колонки родителя в каждой строке дочерней коллекции: чем шире родитель
#   и больше детей, тем больше лишнего трафика и памяти.
# * Две коллекции в одном запросе дают декартово произведение (строк станет n*m).
# * LIMIT/OFFSET поверх joined-коллекции заставляет SQLAlchemy оборачивать запрос в
#   подзапрос, иначе лимит считался бы по строкам JOIN, а не по пилотам.
# joinedload оправдан, когда коллекции маленькие и нужно минимум обращений к БД (один запрос).


без unique(): The unique() method must be invoked on this Result, as it contains results that include joined eager loads against collections
lazy (N+1): 6 | selectinload: 2 | joinedload: 1
самолётов загружено joinedload: 10


## Итоги семинара

1. **psycopg:** соединения через `with`, курсоры через `with`, всегда параметризация `%s`.
2. **Транзакции:** `with conn:` → commit/rollback; короткие транзакции; DDL транзакционен.
3. **SQL-инъекции:** f-string в SQL — уязвимо; `%s` и `:name` — безопасно.
4. **executemany:** один вызов вместо N `execute`; для миллионов строк — `COPY`.
5. **SQLAlchemy Core:** `text()` + `engine.connect()` + `executemany` из списка словарей.
6. **Pandas:** `read_sql(chunksize=…)` и `np.where` — векторизация; PostgreSQL для JOIN, pandas — для фич.
7. **ORM:** `DeclarativeBase`, `relationship(secondary=…)`, `ondelete='CASCADE'` на уровне БД.
8. **N+1:** ловится `echo=True`, лечится `selectinload` (обычно) или `joinedload().unique()` (когда важна экономия на количестве запросов).
